# LiveKit + LangSmith

This notebook shows where LangSmith tracing attaches to a LiveKit voice agent. Local console launch details stay in `voice_demo.workshop`.

## 1. Agent Setup

The LiveKit cascade is AssemblyAI STT, OpenAI LLM, Cartesia TTS, and one LiveKit-native weather tool.

In [ ]:
import os

from dotenv import load_dotenv
from livekit import agents
from livekit.agents import Agent, AgentSession, TurnHandlingOptions, function_tool, room_io
from livekit.plugins import assemblyai, cartesia, openai as lk_openai, silero
from livekit.plugins.turn_detector.multilingual import MultilingualModel

from voice_demo.prompts import GREETING, SYSTEM_PROMPT
from voice_demo.weather import fetch_weather
from voice_demo.workshop import run_livekit_console

load_dotenv()

PROJECT = "voice-workshop-livekit"
LLM_MODEL = os.getenv("LIVEKIT_LLM_MODEL", "gpt-4o-mini")
STT_MODEL = os.getenv("LIVEKIT_STT_MODEL") or None
TTS_VOICE = os.getenv("LIVEKIT_TTS_VOICE") or None
TTS_MODEL = os.getenv("LIVEKIT_TTS_MODEL") or None

assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY before running this notebook."
assert os.getenv("ASSEMBLYAI_API_KEY"), "Set ASSEMBLYAI_API_KEY before running this notebook."
assert os.getenv("CARTESIA_API_KEY"), "Set CARTESIA_API_KEY before running this notebook."
assert os.getenv("LANGSMITH_API_KEY"), "Set LANGSMITH_API_KEY before running this notebook."


In [ ]:
def build_session() -> AgentSession:
    tts_kwargs = {}
    if TTS_MODEL:
        tts_kwargs["model"] = TTS_MODEL
    if TTS_VOICE:
        tts_kwargs["voice"] = TTS_VOICE
    return AgentSession(
        stt=assemblyai.STT(model=STT_MODEL) if STT_MODEL else assemblyai.STT(),
        llm=lk_openai.LLM(model=LLM_MODEL, temperature=0.3),
        tts=cartesia.TTS(**tts_kwargs),
        vad=silero.VAD.load(),
        turn_handling=TurnHandlingOptions(turn_detection=MultilingualModel()),
    )


class Assistant(Agent):
    def __init__(self) -> None:
        super().__init__(instructions=SYSTEM_PROMPT)

    @function_tool
    async def lookup_weather(self, city: str) -> dict:
        """Get the current weather for a single city. Call once per city."""
        return await fetch_weather(city)


## 2. Tracing Setup

`configure_livekit` installs the LangSmith span processor for LiveKit's native OpenTelemetry spans. The `set_thread_id` call groups spans from one LiveKit job into one LangSmith thread.

In [ ]:
from langsmith.integrations.livekit import configure_livekit, set_thread_id

audio_file_path = None
processor = configure_livekit(
    audio_path_provider=lambda: audio_file_path,
    project=PROJECT,
)


## 3. Framework Wiring

This is where the traced LiveKit session starts. The rest of the console runner is hidden in `run_livekit_console`.

In [ ]:
server = agents.AgentServer()


@server.rtc_session()
async def entrypoint(ctx: agents.JobContext) -> None:
    global audio_file_path
    set_thread_id(ctx.job.id)
    audio_file_path = ctx.session_directory / "audio.ogg"

    session = build_session()
    await session.start(
        room=ctx.room,
        agent=Assistant(),
        room_options=room_io.RoomOptions(),
        record={"audio": True},
    )
    await session.say(GREETING)


## 4. Run

This hands control to LiveKit console mode. Stop the process/cell to end the voice session.

In [ ]:
run_livekit_console(server)